# Notebook 6 — Expanded ML design: 5-HT₂A & Dopamine D₃

Implements the locked experimental design. Descriptive chapter untouched; this is the ML case study only.

**Design:** 2 targets · all-phase Landrum-curated sets · **3 representations** (ECFP4, DescriptaStorus,
Fused) · **2 models** (XGBoost, RF) · **2 baselines** (1-NN read-across, CV-Ridge) · validation =
**scaffold split** across the whole grid **+ LOO** on the headline (Fused+XGB) · controls =
**y-randomisation**, **applicability domain**, **learning curve** on the headline · metrics = R², RMSE,
MAE, Spearman ρ. Hyperparameters fixed. Requires internet for Part 1.

## 0. Config & imports

In [ ]:
import os, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs, RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.Scaffolds import MurckoScaffold
from descriptastorus.descriptors import rdNormalizedDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, root_mean_squared_error as rmse, mean_absolute_error as mae
from scipy.stats import spearmanr
try:
    from xgboost import XGBRegressor; HAVE_XGB=True
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor; HAVE_XGB=False; print('xgboost missing -> HistGB')
TARGETS={'5-HT2A':'P28223','Dopamine D3':'P35462'}
BOOST='XGB' if HAVE_XGB else 'GB'
def boost():
    return XGBRegressor(n_estimators=400,max_depth=4,learning_rate=0.05,subsample=0.8,
                        colsample_bytree=0.6,random_state=0,n_jobs=-1) if HAVE_XGB else HistGradientBoostingRegressor(random_state=0)
def rf(): return RandomForestRegressor(n_estimators=400,random_state=0,n_jobs=-1)
print('boosting =',BOOST)

## 1. Fetch all-phase raw Kᵢ (5-HT₂A, D₃)

In [ ]:
from chembl_webresource_client.new_client import new_client
tapi,aapi,mapi=new_client.target,new_client.activity,new_client.molecule
def resolve(acc):
    h=list(tapi.filter(target_components__accession=acc,target_type='SINGLE PROTEIN',organism='Homo sapiens').only(['target_chembl_id']))
    return h[0]['target_chembl_id'] if h else None
def fetch_all(name,acc):
    tid=resolve(acc); print(name,tid)
    a=aapi.filter(target_chembl_id=tid,standard_type='Ki',assay_type='B',pchembl_value__isnull=False,
                  target_organism='Homo sapiens').only(['molecule_chembl_id','canonical_smiles','pchembl_value'])
    raw=pd.DataFrame(list(a)); ids=raw.molecule_chembl_id.dropna().unique().tolist(); ph={}
    for i in range(0,len(ids),200):
        for m in mapi.filter(molecule_chembl_id__in=ids[i:i+200]).only(['molecule_chembl_id','max_phase']): ph[m['molecule_chembl_id']]=m['max_phase']
    raw['max_phase']=pd.to_numeric(raw.molecule_chembl_id.map(ph),errors='coerce'); raw['target']=name
    print(f'  {len(raw)} raw Ki, {raw.molecule_chembl_id.nunique()} molecules'); return raw
raw_all=pd.concat([fetch_all(n,a) for n,a in TARGETS.items()],ignore_index=True)

## 2. Curate + Landrum (uniform) → report preclinical split

In [ ]:
sr=SaltRemover(); pep=Chem.MolFromSmarts('[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)')
def parent(s):
    m=Chem.MolFromSmiles(str(s)) if isinstance(s,str) else None
    if m is None: return None
    p=sr.StripMol(m,dontRemoveEverything=True)
    if p is None or p.GetNumAtoms()==0:
        fr=Chem.GetMolFrags(m,asMols=True); p=max(fr,key=lambda x:x.GetNumAtoms()) if fr else m
    return p
def landrum(v):
    v=pd.Series(v).round(2); v=v[~v.duplicated(keep='first')].tolist()
    v=[x for x in v if not any(abs(abs(x-o)-3.0)<1e-6 for o in v if o!=x)]
    return max(v) if v else np.nan
d=raw_all.copy(); d['pKi']=pd.to_numeric(d.pchembl_value,errors='coerce'); d=d.dropna(subset=['pKi'])
d['mol']=d.canonical_smiles.apply(parent); d=d[d.mol.notna()]
d=d[d.mol.apply(lambda m: Descriptors.MolWt(m))<700]
d=d[~d.mol.apply(lambda m: m.HasSubstructMatch(pep))]
d['parent_smiles']=d.mol.apply(Chem.MolToSmiles)
g=d.groupby(['molecule_chembl_id','target'])
agg=pd.DataFrame({'pKi':g['pKi'].apply(lambda s: landrum(s.tolist())),
                  'parent_smiles':g['parent_smiles'].first(),'max_phase':g['max_phase'].first()}).reset_index().dropna(subset=['pKi'])
agg['preclinical']=~agg.max_phase.isin([1,2,3,4])
for t in TARGETS:
    s=agg[agg.target==t]; print(f'{t}: {len(s)} molecules ({int(s.preclinical.sum())} preclinical + {int((~s.preclinical).sum())} Phase I-IV)')

## 3. Three representations (ECFP4 / DescriptaStorus / Fused) + scaffolds + fps

In [ ]:
gen=rdNormalizedDescriptors.RDKit2DNormalized()
DATA={}
for t in TARGETS:
    sub=agg[agg.target==t].reset_index(drop=True)
    mols=[Chem.MolFromSmiles(s) for s in sub.parent_smiles]; keep=[i for i,m in enumerate(mols) if m]
    sub=sub.iloc[keep].reset_index(drop=True); mols=[mols[i] for i in keep]; N=len(mols)
    fps=[AllChem.GetMorganFingerprintAsBitVect(m,2,nBits=2048) for m in mols]
    E=np.zeros((N,2048),int)
    for i,f in enumerate(fps): DataStructs.ConvertToNumpyArray(f,E[i])
    D=np.nan_to_num(np.array([gen.process(s)[1:] for s in sub.parent_smiles],dtype=float))
    scaf=np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)) for m in mols])
    DATA[t]=dict(N=N,y=sub.pKi.values.astype(float),fps=fps,scaf=scaf,
                 reps={'ECFP4':E,'DescriptaStorus':D,'Fused':np.hstack([E,D])})
    print(t,'N=',N)

## 4. Helpers — scaffold split, metrics, baselines

In [ ]:
def scaffold_split(scaf,N,seed,tf=0.2):
    rng=np.random.RandomState(seed); gr={}
    for i,s in enumerate(scaf): gr.setdefault(s,[]).append(i)
    ks=list(gr); rng.shuffle(ks); nt=int(round(tf*N)); te=[]
    for k in ks:
        if len(te)>=nt: break
        te+=gr[k]
    te=set(te); return [i for i in range(N) if i not in te], sorted(te)
def metrics(yt,yp): return dict(R2=r2_score(yt,yp),RMSE=rmse(yt,yp),MAE=mae(yt,yp),rho=spearmanr(yt,yp).correlation)
def read_across(fps,y,tr,te):
    pred=[]
    for j in te:
        sims=DataStructs.BulkTanimotoSimilarity(fps[j],[fps[i] for i in tr]); pred.append(y[tr][int(np.argmax(sims))])
    return np.array(pred)
def ridge(X,y,tr,te):
    imp=SimpleImputer(strategy='median').fit(X[tr]); Xtr,Xte=imp.transform(X[tr]),imp.transform(X[te])
    sc=StandardScaler().fit(Xtr); Xtr,Xte=sc.transform(Xtr),sc.transform(Xte)
    m=RidgeCV(alphas=np.logspace(-1,4,20)).fit(Xtr,y[tr]); return m.predict(Xte)

## 5. Main grid — scaffold split (25 seeds): 3 reps × 2 models + 2 baselines

In [ ]:
rows=[]
for t in TARGETS:
    dd=DATA[t]; N,y,scaf,fps=dd['N'],dd['y'],dd['scaf'],dd['fps']
    splits=[scaffold_split(scaf,N,s) for s in range(25)]
    # models on each representation
    for rep,X in dd['reps'].items():
        for mn,mk in {BOOST:boost,'RF':rf}.items():
            M={k:[] for k in ['R2','RMSE','MAE','rho']}
            for tr,te in splits:
                if len(te)<5 or len(np.unique(y[te]))<2: continue
                m=mk().fit(X[tr],y[tr]); mm=metrics(y[te],m.predict(X[te]))
                for k in M: M[k].append(mm[k])
            rows.append(dict(target=t,rep=rep,model=mn,n=N,test_n=len(splits[0][1]),
                             R2=round(np.mean(M['R2']),3),R2_lo=round(np.percentile(M['R2'],2.5),2),
                             R2_hi=round(np.percentile(M['R2'],97.5),2),RMSE=round(np.mean(M['RMSE']),3),
                             MAE=round(np.mean(M['MAE']),3),rho=round(np.mean(M['rho']),3)))
    # baselines (ECFP4 read-across; DescriptaStorus CV-Ridge)
    E=dd['reps']['ECFP4']; D=dd['reps']['DescriptaStorus']
    for name,fn in {'1NN read-across':lambda tr,te: read_across(fps,y,tr,te),
                    'CV-Ridge (desc)':lambda tr,te: ridge(D,y,tr,te)}.items():
        r2=[]
        for tr,te in splits:
            if len(te)<5 or len(np.unique(y[te]))<2: continue
            r2.append(r2_score(y[te],fn(tr,te)))
        rows.append(dict(target=t,rep='-',model=name,n=N,test_n=len(splits[0][1]),
                         R2=round(np.mean(r2),3),R2_lo=round(np.percentile(r2,2.5),2),R2_hi=round(np.percentile(r2,97.5),2),
                         RMSE=None,MAE=None,rho=None))
    print('grid done:',t)
grid=pd.DataFrame(rows); print('\n'+grid.to_string(index=False)); grid.to_csv('ml_grid_scaffold.csv',index=False)

## 6. LOO on the headline (Fused + boosting) per target

In [ ]:
loo_rows=[]
for t in TARGETS:
    dd=DATA[t]; X,y=dd['reps']['Fused'],dd['y']; pred=np.zeros(len(y))
    for tr,te in LeaveOneOut().split(X):
        m=boost().fit(X[tr],y[tr]); pred[te]=m.predict(X[te])
    mm=metrics(y,pred); loo_rows.append(dict(target=t,**{k:round(v,3) for k,v in mm.items()}))
    print(t,'LOO done')
loo=pd.DataFrame(loo_rows); print('\nLOO (Fused + '+BOOST+'):'); print(loo.to_string(index=False))

## 7. Controls — y-randomisation + applicability domain (headline)

In [ ]:
ctrl=[]
for t in TARGETS:
    dd=DATA[t]; X,y,scaf,N,fps=dd['reps']['Fused'],dd['y'],dd['scaf'],dd['N'],dd['fps']
    def mean_scaf_r2(yv,nsp=15):
        r=[]
        for s in range(nsp):
            tr,te=scaffold_split(scaf,N,s)
            if len(te)<5 or len(np.unique(yv[te]))<2: continue
            m=boost().fit(X[tr],yv[tr]); r.append(r2_score(yv[te],m.predict(X[te])))
        return np.mean(r)
    true=mean_scaf_r2(y); rng=np.random.RandomState(1)
    null=np.array([mean_scaf_r2(rng.permutation(y)) for _ in range(50)]); p=(np.sum(null>=true)+1)/(len(null)+1)
    # applicability domain: median nearest-neighbour Tanimoto over the scaffold splits
    nn=[]
    for s in range(15):
        tr,te=scaffold_split(scaf,N,s)
        for j in te: nn.append(max(DataStructs.BulkTanimotoSimilarity(fps[j],[fps[i] for i in tr])))
    ctrl.append(dict(target=t,yrand_p=round(p,4),true_R2=round(true,3),AD_median_Tanimoto=round(np.median(nn),3)))
    print(t,'controls done')
print(pd.DataFrame(ctrl).to_string(index=False))

## 8. Learning curve (headline) — does more data still help?

In [ ]:
import matplotlib.pyplot as plt
os.makedirs('figures',exist_ok=True)
fig,axes=plt.subplots(1,len(TARGETS),figsize=(6*len(TARGETS),4.5))
for ax,t in zip(np.atleast_1d(axes),TARGETS):
    dd=DATA[t]; X,y,scaf,N=dd['reps']['Fused'],dd['y'],dd['scaf'],dd['N']
    fracs=[0.3,0.45,0.6,0.75,0.9]; means=[]
    for fr in fracs:
        r=[]
        for s in range(12):
            tr,te=scaffold_split(scaf,N,s); rng=np.random.RandomState(s)
            k=max(10,int(fr*len(tr))); tr=list(rng.choice(tr,k,replace=False))
            if len(np.unique(y[te]))<2: continue
            m=boost().fit(X[tr],y[tr]); r.append(r2_score(y[te],m.predict(X[te])))
        means.append(np.mean(r))
    ax.plot([int(f*N) for f in fracs],means,'o-'); ax.set_title(f'{t} learning curve')
    ax.set_xlabel('training compounds'); ax.set_ylabel('scaffold-split R²'); ax.axhline(0,color='r',ls='--',lw=.8)
fig.tight_layout(); fig.savefig('figures/learning_curve_expanded.png',dpi=300); plt.show()
print('learning curve -> figures/learning_curve_expanded.png')

## 9. Summary
- **`ml_grid_scaffold.csv`** — the full comparison (3 reps × 2 models + 2 baselines, both targets), scaffold split.
- **LOO** table — headline (Fused + boosting) per target.
- **Controls** — y-randomisation p and applicability-domain Tanimoto.
- **Learning curve** — if still rising at full n, adding preclinical data demonstrably helped.
- Report R², RMSE, MAE, ρ; explain in the text that a narrow pKᵢ range depresses R² even when RMSE is
  good. **No multi-target panel** (removed by design).